In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import timm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from collections import Counter
import numpy as np

# 1. Device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Transforms (for RGB ViT)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

# 3. Dataset and class names
dataset_path = "../dataset"
full_dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
class_names = full_dataset.classes
num_classes = len(class_names)

# 4. Train/Val split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)

print(f"\n📊 Dataset Summary:")
print(f"Total samples     : {len(full_dataset)}")
print(f"Training samples  : {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Number of classes : {num_classes} -> {class_names}\n")

# 5. Analyze class distribution in training/val
def print_class_distribution(dataset, dataset_name):
    label_counts = Counter()
    for _, label in dataset:
        label_counts[label] += 1
    print(f"{dataset_name} class distribution:")
    for label, count in sorted(label_counts.items()):
        print(f"  {class_names[label]}: {count}")
    print()

print_class_distribution(train_dataset, "Training")
print_class_distribution(val_dataset, "Validation")

# 6. Load ViT model
model = timm.create_model('vit_base_patch16_224', pretrained=True)
model.head = nn.Linear(model.head.in_features, num_classes)
model = model.to(device)

# 7. Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 8. Training loop with validation
num_epochs = 10
train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    # Validation loop
    model.eval()
    total_val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item()

            # Accuracy tracking
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_train = total_train_loss / len(train_loader)
    avg_val = total_val_loss / len(val_loader)
    val_acc = correct / total

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | Val Accuracy: {val_acc:.4f}")

# 9. Plot training and validation loss
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

# 10. Plot validation accuracy
plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label='Val Accuracy', color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 11. Confusion Matrix on Validation Set
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# 12. Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("Confusion Matrix on Validation Set")
plt.tight_layout()
plt.show()

# 13. Classification Report
print("\n Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))


Using device: cuda:0

📊 Dataset Summary:
Total samples     : 40216
Training samples  : 32172
Validation samples: 8044
Number of classes : 4 -> ['benign_preprocessed', 'pulse_preprocessed', 'single_tone_preprocessed', 'wideband_preprocessed']



KeyboardInterrupt: 

: 